# Du coût d'investissement à l'annuité

*Comment un investissement en €/kW devient un coût annuel en €/kW/an, puis un coût par MWh, et pourquoi ce dernier ne suffit pas à juger un actif.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/cout-annualise.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets.

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), Path.cwd())  # la racine du dépôt des cas
if not (RACINE / "pommes_cases").is_dir():
    raise FileNotFoundError(f"pas de pommes_cases/ sous {RACINE} : placez ce notebook dans le dépôt "
                            "pommes-case-studies (à sa racine ou dans un de ses dossiers)")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")

## Pourquoi annualiser

Une batterie coûte des euros par kW le jour où on l'achète. Elle gagne des euros par kW et par an pendant vingt-cinq ans. Pour comparer les deux, on ramène l'investissement à un coût annuel : l'annuité. C'est ce que fait pommes dans son objectif, et c'est ce que vous lisez dans chaque verdict des cas.

Deux paramètres suffisent : la durée de vie n, en années, et le taux d'actualisation r. Le taux dit ce que vaut un euro de l'an prochain par rapport à un euro d'aujourd'hui. Le cas A emprunte 4 % au TYNDP ; le cas E prend 8 %, « typique d'un projet industriel ». Le même actif n'a donc pas le même coût annuel selon qui le finance.

## Le facteur de recouvrement du capital

L'annuité constante qui rembourse un euro en n ans au taux r vaut :

CRF(r, n) = r / (1 − (1 + r)^−n)

et 1 / n quand r vaut zéro. Le coût annualisé d'un investissement I est I × CRF. On y ajoute le coût fixe d'exploitation, en €/kW/an, qui ne dépend ni du taux ni de la durée.

Le cas A donne le fil rouge : une batterie au coût du TYNDP 2024, millésime 2030. Les hypothèses sont dans `A_batteries/hypotheses.yaml`, statut « repris » : 192 €/kW pour la puissance, 171 €/kWh pour l'énergie, 25 ans, 4 %, 650 €/MW/an de coût fixe. Une batterie de 2 h coûte donc 192 + 2 × 171 = 534 €/kW d'investissement.

In [ ]:
def crf(r, n):
    return r / (1 - (1 + r) ** -n) if r else 1 / n

cp, ce, fixe = 192, 171, 650 / 1000          # €/kW, €/kWh, €/kW/an
print(round(crf(0.04, 25), 4))               # 0.064
for h in (1, 2, 4):
    print(h, "h :", round(crf(0.04, 25) * (cp + ce * h) + fixe, 1), "€/kW/an")
# 1 h : 23.9 ; 2 h : 34.8 ; 4 h : 56.7
for r, n in ((0.08, 25), (0.04, 15), (0.08, 15)):
    print(r, n, [round(crf(r, n) * (cp + ce * h) + fixe, 1) for h in (1, 2, 4)])
# 8 %, 25 ans : [34.7, 50.7, 82.7]
# 4 %, 15 ans : [33.3, 48.7, 79.4]
# 8 %, 15 ans : [43.1, 63.0, 103.0]

Les trois valeurs du README du cas A sont retrouvées : 23,9, 34,8 et 56,7 €/kW/an pour 1, 2 et 4 h. Passer de 4 % à 8 % fait grimper la batterie de 2 h de 34,8 à 50,7 €/kW/an, soit 46 % de plus. Passer de 25 à 15 ans la porte à 48,7. Les deux ensemble la portent à 63,0, presque le double. Or son revenu d'arbitrage en 2030 vaut 25,7 €/kW/an dans le TYNDP 2026 NT+ et va de 4,7 à 48,2 dans le jeu POMMES. Le taux et la durée de vie décident du verdict autant que les prix.

## Comment pommes annualise

Vous n'écrivez pas la formule dans un cas : pommes la calcule. Chaque composant de pommes_craft porte un coût d'investissement (`invest_cost`, ou `invest_cost_power` et `invest_cost_energy` pour un stockage), une durée de vie `life_span` et un taux `finance_rate`. En construisant le programme linéaire, pommes évalue `pommes.utils.crf(finance_rate, n)` et multiplie par l'investissement, où n vaut en fait `year_dec − year_inv`, d'où le réglage ci-dessous. Le résultat est le paramètre `storage_annuity_cost_power` pour la puissance d'un stockage, `storage_annuity_cost_energy` pour l'énergie, `conversion_annuity_cost` pour une chaudière ou une centrale. L'objectif additionne ces annuités, les coûts fixes et les coûts variables.

Le bloc suivant met la batterie du cas A dans une zone au prix exogène et lit le paramètre sans résoudre :

In [ ]:
import sys, numpy as np
from pommes_cases.blocks import price_zone_model
from pommes_cases.solve import build_lp
from pommes_craft import StorageTechnology

model, zone = price_zone_model("verif", {2030: np.full(8760, 50.0)}, year_dec=2055)
with model.context():
    zone.add_component(StorageTechnology(
        name="batterie", factor_in={"electricity": -1.0}, factor_out={"electricity": 0.91},
        factor_keep={"electricity": 0.0}, dissipation=0.0, energy_to_power_ratio=2.0,
        invest_cost_power=192_000.0, invest_cost_energy=171_000.0, fixed_cost_power=650.0,
        finance_rate=0.04, life_span=25.0,
        power_capacity_investment_min=0.0, power_capacity_investment_max=1.0))
lp, params = build_lp(model)
ap = float(params["storage_annuity_cost_power"].squeeze())   # 12290.3 €/MW/an = 192000 × 0.064
ae = float(params["storage_annuity_cost_energy"].squeeze())  # 10946.0 €/MWh/an
print(round((ap + 2 * ae) / 1000 + 0.65, 1))                 # 34.8 €/kW/an

Deux détails comptent. D'abord, pommes n'ouvre une capacité que si l'écart entre `year_dec` et l'année d'investissement égale la durée de vie : d'où `year_dec=2055`. Ensuite, `EconomicHypothesis` porte quatre réglages : `discount_rate`, qui actualise les années d'exploitation vers `year_ref`, `planning_step`, le pas entre deux années de planification, et `operation_year_weight`, le poids de chaque année d'exploitation, 1 par défaut. Les cas du dépôt posent `discount_rate=0.0` et une seule année d'investissement : l'objectif est le coût d'une année, annuités comprises, et les €/kW/an se lisent directement. Le cas A empile plusieurs séries de prix comme autant d'années d'exploitation, aux mêmes capacités ; le cas E n'en a qu'une. Le taux de 4 % est celui du composant, `finance_rate`, pas celui de l'actualisation. Les poids d'année sont décrits dans [Model description (1) : objectif, actualisation, annuités, poids d'année et conditions d'exploitation](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html).

## Du €/kW/an au €/MWh : le LCOE

Le coût moyen actualisé d'un MWh, ou LCOE, divise le coût annuel par l'énergie produite dans l'année. Pour un actif de 1 MW qui tourne au facteur de charge f, cette énergie vaut 8 760 × f MWh. Le capital pèse donc d'autant plus que l'actif tourne peu.

Le cas E le montre avec une chaudière électrique. `E_vapeur/hypotheses.yaml` donne 220 000 €/MW thermique, 1 375 €/MW/an de coût fixe, 0,642 €/MWh de coût variable, 25 ans, un rendement de 0,99, statut « vérifié » depuis les fiches de la Danish Energy Agency, et un taux de 8 %, statut « choix du cas ». La demande vaut 7 300 MWh/an à 1,25 MW, deux postes de 8 h.

In [ ]:
a = crf(0.08, 25)                      # 0.0937
mwh_par_mw = 7300 / 1.25               # 5840 MWh/MW/an, facteur de charge 0.667
cap_elec = 220_000 * a + 1_375         # 21984 €/MW/an
print(round(cap_elec / mwh_par_mw, 2))                      # 3.76 €/MWh de vapeur
for f in (8 / 24, 16 / 24, 1.0):
    print(round(f, 2), round(cap_elec / (8760 * f), 2))     # 7.53 ; 3.76 ; 2.51
prix_moyen = 60.0                      # €/MWh, choix d'illustration
print(round(cap_elec / mwh_par_mw + 0.642 + prix_moyen / 0.99, 1))   # 65.0 €/MWh
gaz = (22.6 + 0.202 * 45) / 0.94       # gaz TYNDP 2024 en 2030 + CO2 ETS2 à 45 €/t : 33.71 €/MWh
print(round((90_000 * a + 2_000) / mwh_par_mw + 1.23 + gaz, 1))      # 36.7 €/MWh

Deux lectures. Pour une chaudière, le capital est petit : 3,76 €/MWh de vapeur sur un total de l'ordre de 60 €/MWh. L'électricité achetée fait presque tout, et le verdict dépend du prix horaire, pas du CRF. La dernière ligne retrouve les 36,7 €/MWh de la chaudière à gaz seule du README du cas E, sous le TYNDP 2024 en 2030 : le LCOE d'un actif qui tourne à heures fixes se calcule à la main, sans pommes.

## Les limites du LCOE

Le LCOE répond à « combien coûte un MWh produit », pas à « combien vaut ce MWh ». Trois pièges.

Le facteur de charge n'est pas une donnée. Pour une batterie ou une chaudière hybride, c'est le résultat de l'optimisation face aux prix. Dans le cas E, l'hybride produit 0 à 67 % de sa vapeur à l'électricité selon le jeu de prix, pour 24 à 43 €/MWh.

La valeur de marché dépend des heures. Deux actifs au même LCOE ne gagnent pas la même chose s'ils produisent à des heures différentes. Dans le cas A, la série centrale de fos-sur-mer rapporte 54,8 €/kW/an, dont 20,9 en 15 heures au plafond de 1 000 €/MWh, soit 38 % du revenu. Sans ces heures, la batterie est réoptimisée (chaque heure au plafond est ramenée au plus haut prix hors plafond de l'année) et rapporte 34,8 €/kW/an — pas 54,8 − 20,9 = 33,9 (`A_batteries/results.json`, lignes `sans_plafond` à `false` et `true`). 34,8 €/kW/an, c'est justement le coût annualisé d'une batterie de 2 h au coût central du TYNDP 2030 : l'équilibre exact.

La cannibalisation. Quand toutes les batteries déchargent aux mêmes heures, elles font baisser le prix qu'elles visaient. Le cas A le modélise par une pente de marché : à 1 €/MWh par GW déchargé, une flotte de 10 GW gagne environ 20 % de moins par kW qu'une batterie seule, 19,9 contre 25,7 €/kW/an sous le TYNDP 2026. Le LCOE ne bouge pas ; la valeur fond.

## Le coût de défaillance

Un modèle de planification doit aussi décider ce que coûte un MWh non servi. Dans pommes, c'est le paramètre `load_shedding_cost`, nul par défaut quand aucun composant n'est posé ; le composant `LoadShedding` de pommes_craft le fixe par son argument `cost`, 1 000 €/MWh par défaut. Les cas à prix exogène lui donnent une capacité nulle : la demande doit être servie. Dans un modèle de système entier, ce coût fixe la hauteur des pics : le prix dual monte au coût de défaillance quand la demande n'est pas servie. Le cas A traite toute heure à 1 000 €/MWh ou plus comme une heure de défaillance du modèle de prix, statut « choix du cas », et donne le verdict avec et sans ces heures. Pour l'adéquation et la LOLE : Adéquation : défaillance, pointe « à une chance sur dix », LOLE dans le capacity expansion (remis en cours).

## Vous l'avez compris quand…

- Vous retrouvez 23,9, 34,8 et 56,7 €/kW/an à la main depuis les cinq nombres de `A_batteries/hypotheses.yaml`, puis 50,7 €/kW/an pour la batterie de 2 h à 8 %.
- Vous lisez `storage_annuity_cost_power` dans les paramètres pommes d'un cas et vous expliquez pourquoi il vaut 12 290 €/MW/an.
- Vous dites, sans calcul, pourquoi un taux de 8 % pénalise une batterie bien plus qu'une chaudière électrique.
- Vous citez un cas où deux actifs au même coût annuel n'ont pas la même valeur, et vous nommez le mécanisme.

## Pour aller plus loin

- Le coût du capital, les contrats qui le font baisser (tarif d'achat, contrat pour différence, contrat d'achat de long terme, prêt public) et qui paie au bout du compte : la note « Financer la transition » [Financer la transition : le coût du capital, les contrats, et qui paie au bout du compte (note, 22/11/2026)](https://www.robingirard.eu/assets/parcours-es/latest/notes/note-financer-transition.html). Passer de 3 % à 9 % y multiplie le coût du mégawattheure par 1,6 pour l'éolien et le solaire, et par 2,7 pour le nucléaire neuf.
- L'objectif de pommes, l'actualisation φ1, le facteur φ2 et les poids d'année : [Model description (1) : objectif, actualisation, annuités, poids d'année et conditions d'exploitation](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html).
- Le LCOE décomposé en part capital et part variable, par famille de moyens : [Évaluer le coût économique de la production d'électricité (décomposition du LCOE)](https://www.energy-alternatives.eu/2020/08/20/decomposition-lcoe.html).
- Les planches du cours sur le CRF, le LCOE et les courbes de coût, remises en cours : Coûts : taux d'actualisation, CRF, coûts annualisés, LCOE et ses limites, coût du nucléaire, courbes de coût éolien/PV, batteries, électrolyseurs (remis en cours).
- Un revenu par kW face au coût annuel : [Cas A — Batteries : que rapporte l'arbitrage au n-ième GW ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/A_batteries) ; une annuité face à un prix horaire : [Cas E — Vapeur à 180 °C : tout électrifier, garder le gaz, ou les deux ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/E_vapeur).

## Sources

1. [Dépôt pommes-case-studies : A_batteries/hypotheses.yaml, A_batteries/README.md, E_vapeur/hypotheses.yaml, E_vapeur/README.md](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)
2. [Documentation pommes, Model description : objectif, actualisation, annuités](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html)
3. [Documentation pommes, référence API (pommes.utils.crf)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/autoapi/index.html)
4. [Évaluer le coût économique de la production d'électricité (décomposition du LCOE)](https://www.energy-alternatives.eu/2020/08/20/decomposition-lcoe.html)